In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Polarisation Assessment

## Training a K=4 LDA model on the full wave data

In [2]:
import os
import joblib
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation
import json

# Set fixed number of topics per wave (K=4 across all)
fixed_k = 4

# List of waves
waves = [4, 5, 6, 7]

# Store results
k4_top_issues_by_wave = {}

# Path to your encoded CSVs
encoded_data_path = "../data/processed/2_lda_encoded"

# Where to save the full-data LDA models
model_save_dir = "../data/lda_k4_full_models"
os.makedirs(model_save_dir, exist_ok=True)

#Where to save top issues
top_issues_save_dir = "../data/reports/top_issues"

# Where to save k4 top issues
top_issues_save_dir = "../data/reports/top_issues"

# Metadata columns to drop for training
drop_cols = ["country", "year", "wave", "sex", "age_6", "education_l"]

for wave in waves:
    print(f"\n🌊 Training full K=4 LDA model for Wave {wave} with K={fixed_k}")

    # Load encoded issue data
    df = pd.read_csv(f"{encoded_data_path}/wvs_wave{wave}.csv")
    lda_data = df.drop(columns=drop_cols)

    # Fit LDA model on full data
    lda_model = LatentDirichletAllocation(
        n_components=4,
        doc_topic_prior=0.25,
        topic_word_prior=0.1,
        learning_method='online',
        learning_decay=0.7,
        learning_offset=10.0,
        max_iter=20,
        batch_size=1000,
        evaluate_every=-1,
        mean_change_tol=0.001,
        max_doc_update_iter=100,
        n_jobs=-1,
        random_state=42
    )
    lda_model.fit(lda_data)


 # Extract topic-word matrix and normalize
    topic_words = pd.DataFrame(lda_model.components_, columns=lda_data.columns)
    topic_words = topic_words.div(topic_words.sum(axis=1), axis=0)
    topic_words = topic_words.T
    topic_words.columns = [f"Ideology_{i+1}" for i in range(fixed_k)]
    
    # Get top 10 issues per ideology type
    top_issues = topic_words.apply(lambda x: x.nlargest(10).index.tolist(), axis=0)
  
    # Save k4 top issues
    k4_top_issues_by_wave[wave] = top_issues
    
    print(f"Saved top 10 issues for each ideology in Wave {wave}")

    # Save the model
    model_path = os.path.join(model_save_dir, f"lda_wave{wave}_K4.pkl")
    joblib.dump(lda_model, model_path)
    print(f"💾 Saved full K=4 model for Wave {wave} at: {model_path}")


🌊 Training full K=4 LDA model for Wave 4 with K=4
Saved top 10 issues for each ideology in Wave 4
💾 Saved full K=4 model for Wave 4 at: ../data/lda_k4_full_models/lda_wave4_K4.pkl

🌊 Training full K=4 LDA model for Wave 5 with K=4
Saved top 10 issues for each ideology in Wave 5
💾 Saved full K=4 model for Wave 5 at: ../data/lda_k4_full_models/lda_wave5_K4.pkl

🌊 Training full K=4 LDA model for Wave 6 with K=4
Saved top 10 issues for each ideology in Wave 6
💾 Saved full K=4 model for Wave 6 at: ../data/lda_k4_full_models/lda_wave6_K4.pkl

🌊 Training full K=4 LDA model for Wave 7 with K=4
Saved top 10 issues for each ideology in Wave 7
💾 Saved full K=4 model for Wave 7 at: ../data/lda_k4_full_models/lda_wave7_K4.pkl


In [40]:
import json
import os

# Defining output directory
output_dir = '../data/reports/top_issues'
os.makedirs(output_dir, exist_ok=True)

# Load your variable dictionary
with open("variable_dict.json", "r") as f:
    variable_dict = json.load(f)

# Extend the dictionary to include _support and _oppose
extended_dict = {}
for var, desc in variable_dict.items():
    extended_dict[f"{var}_support"] = f"{desc} (Support)"
    extended_dict[f"{var}_oppose"] = f"{desc} (Oppose)"

# Function to apply the mapping to a top_issues DataFrame
def map_features_to_labels(top_issues_df):
    return top_issues_df.applymap(lambda x: extended_dict.get(x, x))  # fallback to original if not found

for wave, top_issues_df in k4_top_issues_by_wave.items():
    # Map features to descriptions
    labeled_issues = map_features_to_labels(top_issues_df)
    # Save labeled issues to CSV for reporting
    output_path = os.path.join(output_dir, f"k4_wave{wave}_top_issues_labeled.csv")
    labeled_issues.to_csv(output_path, index=False)
    print(f"✅ Labeled top issues saved for Wave {wave}")

✅ Labeled top issues saved for Wave 4
✅ Labeled top issues saved for Wave 5
✅ Labeled top issues saved for Wave 6
✅ Labeled top issues saved for Wave 7


/var/folders/vk/c6csf7ws6pj9wy406twfjkfw0000gn/T/ipykernel_54934/3769922505.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return top_issues_df.applymap(lambda x: extended_dict.get(x, x))  # fallback to original if not found


### Tables for ideology types with fixed k=4

In [41]:
import pandas as pd

# Define the paths to your CSV files
files = {
    4: "../data/reports/top_issues/k4_wave4_top_issues_labeled.csv",
    5: "../data/reports/top_issues/k4_wave5_top_issues_labeled.csv",
    6: "../data/reports/top_issues/k4_wave6_top_issues_labeled.csv",
    7: "../data/reports/top_issues/k4_wave7_top_issues_labeled.csv"
}

# Load the CSV files into a dictionary of DataFrames, keyed by wave number.
top_issues_by_wave = {wave: pd.read_csv(path) for wave, path in files.items()}


In [42]:
for wave, df in top_issues_by_wave.items():
    print(f"Top Issues for Wave {wave}")
    display(df)  # In Jupyter Notebook, 'display' will show a nicely formatted table.

Top Issues for Wave 4


,Ideology_1,Ideology_2,Ideology_3,Ideology_4
0,Suicide – justifiable? (Oppose),Confidence in the Church (Oppose),Homosexuals as neighbors (Oppose),Homosexuality – justifiable? (Oppose)
1,Abortion – justifiable? (Oppose),Confidence in Parliament (Oppose),People with AIDS as neighbors (Oppose),Prostitution – justifiable? (Oppose)
2,Prostitution – justifiable? (Oppose),Confidence in Civil Services (Oppose),Divorce – justifiable? (Support),Suicide – justifiable? (Oppose)
3,Someone accepting a bribe – justifiable? (Oppose),Confidence in Major Companies (Oppose),Immigrants foreign workers as neighbors (Oppose),Confidence in the Police (Support)
4,Cheating on taxes – justifiable? (Oppose),Confidence in the Police (Oppose),Different race as neighbors (Oppose),Confidence in Civil Services (Support)
5,Euthanasia – justifiable? (Oppose),Confidence in Armed Forces (Oppose),National pride (Support),National pride (Support)
6,Avoiding a fare on public transport – justifia...,Confidence in Labour Unions (Oppose),Euthanasia – justifiable? (Support),Someone accepting a bribe – justifiable? (Oppose)
7,Homosexuality – justifiable? (Oppose),Confidence in the Press (Oppose),Abortion – justifiable? (Support),Abortion – justifiable? (Oppose)
8,National pride (Support),Preference for native workers in job scarcity ...,Homosexuality – justifiable? (Support),Confidence in Parliament (Support)
9,Justifiable: Claiming government benefits to w...,Someone accepting a bribe – justifiable? (Oppose),Someone accepting a bribe – justifiable? (Oppose),Cheating on taxes – justifiable? (Oppose)


Top Issues for Wave 5


,Ideology_1,Ideology_2,Ideology_3,Ideology_4
0,Immigrants foreign workers as neighbors (Oppose),Confidence in Parliament (Oppose),Euthanasia – justifiable? (Support),People with AIDS as neighbors (Support)
1,Suicide – justifiable? (Oppose),Confidence in Civil Services (Oppose),Divorce – justifiable? (Support),Homosexuals as neighbors (Support)
2,Cheating on taxes – justifiable? (Oppose),Confidence in Labour Unions (Oppose),Homosexuality – justifiable? (Support),Homosexuality – justifiable? (Oppose)
3,Someone accepting a bribe – justifiable? (Oppose),Confidence in Justice System (Oppose),Abortion – justifiable? (Support),Drug addicts as neighbors (Support)
4,Different race as neighbors (Oppose),Confidence in the Press (Oppose),Different race as neighbors (Oppose),Immigrants foreign workers as neighbors (Support)
5,Avoiding a fare on public transport – justifia...,Confidence in the Police (Oppose),Immigrants foreign workers as neighbors (Oppose),Prostitution – justifiable? (Oppose)
6,Prostitution – justifiable? (Oppose),Confidence in Major Companies (Oppose),Homosexuals as neighbors (Oppose),National pride (Support)
7,Justifiable: Claiming government benefits to w...,Someone accepting a bribe – justifiable? (Oppose),People with AIDS as neighbors (Oppose),Confidence in Armed Forces (Support)
8,National pride (Support),Suicide – justifiable? (Oppose),National pride (Support),Preference for native workers in job scarcity ...
9,Abortion – justifiable? (Oppose),Different race as neighbors (Oppose),Drug addicts as neighbors (Support),Abortion – justifiable? (Oppose)


Top Issues for Wave 6


,Ideology_1,Ideology_2,Ideology_3,Ideology_4
0,Confidence in Parliament (Oppose),Confidence in Armed Forces (Oppose),Divorce – justifiable? (Support),Confidence in Justice System (Support)
1,Someone accepting a bribe – justifiable? (Oppose),Confidence in Justice System (Oppose),Homosexuals as neighbors (Oppose),Confidence in Civil Services (Support)
2,Prostitution – justifiable? (Oppose),Confidence in the Police (Oppose),Confidence in the Police (Support),Confidence in the Police (Support)
3,Suicide – justifiable? (Oppose),Divorce – justifiable? (Support),Different race as neighbors (Oppose),Confidence in Armed Forces (Support)
4,Cheating on taxes – justifiable? (Oppose),Confidence in Parliament (Oppose),Homosexuality – justifiable? (Support),Prostitution – justifiable? (Oppose)
5,Euthanasia – justifiable? (Oppose),Confidence in Civil Services (Oppose),People with AIDS as neighbors (Oppose),Euthanasia – justifiable? (Oppose)
6,Avoiding a fare on public transport – justifia...,Confidence in the Press (Oppose),Immigrants foreign workers as neighbors (Oppose),Suicide – justifiable? (Oppose)
7,Abortion – justifiable? (Oppose),Avoiding a fare on public transport – justifia...,Confidence in Justice System (Support),National pride (Support)
8,Justifiable: Claiming government benefits to w...,Confidence in Labour Unions (Oppose),Abortion – justifiable? (Support),Abortion – justifiable? (Oppose)
9,Confidence in Labour Unions (Oppose),Different race as neighbors (Oppose),National pride (Support),Confidence in the Press (Support)


Top Issues for Wave 7


,Ideology_1,Ideology_2,Ideology_3,Ideology_4
0,Prostitution – justifiable? (Oppose),Confidence in Parliament (Oppose),Divorce – justifiable? (Support),Immigrants foreign workers as neighbors (Support)
1,Confidence in Armed Forces (Support),Confidence in Civil Services (Oppose),Homosexuals as neighbors (Oppose),Different race as neighbors (Support)
2,Suicide – justifiable? (Oppose),Confidence in Justice System (Oppose),Homosexuality – justifiable? (Support),Avoiding a fare on public transport – justifia...
3,Confidence in the Police (Support),Confidence in the Press (Oppose),Different race as neighbors (Oppose),Justifiable: Claiming government benefits to w...
4,Confidence in Justice System (Support),Confidence in the Police (Oppose),Euthanasia – justifiable? (Support),People with AIDS as neighbors (Support)
5,Someone accepting a bribe – justifiable? (Oppose),Confidence in Labour Unions (Oppose),Immigrants foreign workers as neighbors (Oppose),Cheating on taxes – justifiable? (Support)
6,Abortion – justifiable? (Oppose),Suicide – justifiable? (Oppose),People with AIDS as neighbors (Oppose),Homosexuals as neighbors (Support)
7,Cheating on taxes – justifiable? (Oppose),Confidence in Major Companies (Oppose),Someone accepting a bribe – justifiable? (Oppose),Someone accepting a bribe – justifiable? (Supp...
8,National pride (Support),Someone accepting a bribe – justifiable? (Oppose),Abortion – justifiable? (Support),Confidence in the Church (Support)
9,Confidence in Civil Services (Support),Prostitution – justifiable? (Oppose),Drug addicts as neighbors (Support),Divorce – justifiable? (Support)


## Measure Polarisation

In [ ]:
import os
import joblib
import pandas as pd

# Folder paths
encoded_data_path = "../data/processed/2_lda_encoded"
model_path = "../data/lda_k4_full_models"
output_path = "../data/3_theta_with_metadata"
os.makedirs(output_path, exist_ok=True)

# Metadata columns
metadata_cols = ["country", "year", "wave", "sex", "age_6", "education_l"]

# Store final results
all_waves_df = []

for wave in [4, 5, 6, 7]:
    print(f"\n🔄 Processing Wave {wave}")

    # Load encoded data for each wave
    df = pd.read_csv(f"{encoded_data_path}/wvs_wave{wave}.csv")
    metadata = df[metadata_cols].copy()   
    lda_input = df.drop(columns=metadata_cols)  

    # Load K=4 model
    model_file = f"{model_path}/lda_wave{wave}_K4.pkl"  # Which was previously saved
    lda_model = joblib.load(model_file)

    # Transform data into topic space
    # From scikit-learn’s LatentDirichletAllocation (LDA) class
    # theta is the document-topic distribution matrix
    theta = lda_model.transform(lda_input) 
    theta_df = pd.DataFrame(theta, columns=[f"theta_{i+1}" for i in range(4)])

    # Merge metadata + topic shares
    wave_df = pd.concat([metadata, theta_df], axis=1)
    all_waves_df.append(wave_df)

# Concatenate all waves
final_df = pd.concat(all_waves_df, ignore_index=True)

# Save result
final_df.to_csv(f"{output_path}/all_waves_theta_metadata.csv", index=False)
print("\n✅ Saved combined θ + metadata for all waves!")



🔄 Processing Wave 4

🔄 Processing Wave 5

🔄 Processing Wave 6

🔄 Processing Wave 7

✅ Saved combined θ + metadata for all waves!


In [16]:
import numpy as np
import pandas as pd
from scipy.stats import zscore


# Define the output path if not already defined
output_path = "../data/3_theta_with_metadata"

# Load the full dataset (with metadata and theta values) for all waves
df_full_wave = pd.read_csv(f"{output_path}/all_waves_theta_metadata.csv")

# Assign Dominant Ideological Types (based on highest theta value previously retrieved)
df_full_wave["dominant_topic"] = np.argmax(df_full_wave[[f"theta_{i+1}" for i in range(4)]].values, axis=1)

# Compute Group-Level θ̃ (mean topic distribution for each dominant ideological type, grouped by country and year)
group_theta = (
    df_full_wave
    .groupby(["country", "year", "wave", "dominant_topic"])[["theta_1", "theta_2", "theta_3", "theta_4"]]
    .mean()
    .reset_index()
)

# Calculate ρ_tj (the similarity between the topic vectors using correlation)
def calculate_rho(group_t, group_j):
    # Assuming 'group_t' and 'group_j' are the topic vectors (β_t and β_j)
    correlation = np.corrcoef(group_t, group_j)[0, 1]
    return (3 - correlation) / 2  # rho_tj as per the formula


######## Polarization function ########
def compute_polarization(group_df):
    if group_df.shape[0] <= 1:
        return 0.0

    group_sizes = (
    df_full_wave.groupby(["country", "year", "wave", "dominant_topic"])
    .size()
    .reset_index(name="pi")
)

    group_df = pd.merge(group_df, group_sizes, on=["country", "year", "wave", "dominant_topic"])

# Normalize π to population shares within (country, year, wave)
    group_df["pi"] = group_df["pi"] / group_df["pi"].sum()

    nu = 0.5  # Sensitivity parameter
    total_pi = group_df["pi"].sum()
    kappa = total_pi ** (-2 + nu) if total_pi > 0 else 0

    polarization_score = 0
    n = group_df.shape[0]

    theta_values = group_df[theta_cols].values
    pi_values = group_df["pi"].values

    for t in range(n):
        for j in range(n):
            theta_t = theta_values[t]
            theta_j = theta_values[j]
            pi_t = pi_values[t]
            pi_j = pi_values[j]

            rho_tj = calculate_rho(theta_t, theta_j)
            weighted_abs_diff = np.sum(np.abs(theta_t - theta_j) * rho_tj)

            polarization_score += (pi_t ** (1 + nu)) * pi_j * weighted_abs_diff

    return kappa * polarization_score

# Compute polarization per (country, year, wave)
polarization_scores = (
    group_theta
    .groupby(["country", "year", "wave"])
    .apply(compute_polarization)
    .reset_index(name="polarization_score")
)

# Save results
output_dir = "../data/reports/polarisation_scores"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "polarization_scores_by_country_year.csv")
polarization_scores.to_csv(output_file, index=False)

# Compute z-scores for the polarization scores
polarization_scores["polarization_zscore"] = zscore(polarization_scores["polarization_score"])

# Save the updated results with z-scores
output_file_z = os.path.join(output_dir, "polarization_scores_by_country_year_zscored.csv")
polarization_scores.to_csv(output_file_z, index=False)

print(f"✅ Saved polarization scores with z-scores to {output_file_z}")

print(f"✅ Saved polarization scores to {output_file}")

✅ Saved polarization scores with z-scores to ../data/reports/polarisation_scores/polarization_scores_by_country_year_zscored.csv
✅ Saved polarization scores to ../data/reports/polarisation_scores/polarization_scores_by_country_year.csv


/var/folders/vk/c6csf7ws6pj9wy406twfjkfw0000gn/T/ipykernel_58556/2153145346.py:74: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_polarization)
